Step 3 Data Visualization

In [19]:
import osmnx as ox
import networkx as nx
import geopandas as gpd
import pandas as pd
import folium
from folium.plugins import HeatMap
from shapely.geometry import LineString
import branca.colormap as cm

In [5]:
def route_to_gdf(G, route, weight_used):
    """
    Converts a sequence of node IDs (from Step 2) into a LineString GeoDataFrame
    by looking up edges and their corresponding attributes in the scored graph G.
    """
    edge_data = []
    
    # Iterate through consecutive node pairs in the route sequence
    for u, v in zip(route[:-1], route[1:]):
        # Check if the edge exists in the MultiDiGraph
        if G.has_edge(u, v):
            # Access edge data
            data = G.get_edge_data(u, v)[0]
            
            # Extract geometry if it exists, otherwise generate a straight line between nodes
            if 'geometry' in data:
                geom = data['geometry']
            else:
                u_coord = G.nodes[u]
                v_coord = G.nodes[v]
                geom = LineString([(u_coord['x'], u_coord['y']), (v_coord['x'], v_coord['y'])])
            
            # Append geometric and safety properties for this edge
            edge_data.append({
                "geometry": geom,
                "route_type": weight_used,
                "length": data.get("length", 0),
                "safety_cost": data.get("safety_cost", 0),
                "accident_count_50m": data.get("accident_count_50m", 0),
                "accident_risk_norm": data.get("accident_risk_norm", 0)
            })
            
    # Get the Coordinate Reference System (CRS) from graph, default to WGS84 (EPSG:4326)
    graph_crs = G.graph.get('crs', 'EPSG:4326')
    
    # Construct the final GeoDataFrame
    gdf = gpd.GeoDataFrame(edge_data, crs=graph_crs)
    return gdf

In [27]:
def run_step3_visualization(G, shortest_route, safety_route, route_summary, accident_df=None):
    """
    Executes all tasks in Step 3:
    1. Exports the statistical route summary to a CSV file.
    2. Converts node ID lists into spatial GeoDataFrames.
    3. Generates an interactive HTML map containing both routes and an accident heatmap.
    """
    # Export Route Summary to CSV
    csv_filename = "route_summary.csv"
    route_summary.to_csv(csv_filename, index=False)
    print(f"[Step 3] Successfully exported: {csv_filename}")
    
    # Convert Node Sequences to GeoDataFrames
    shortest_gdf = route_to_gdf(G, shortest_route, weight_used="length")
    safety_gdf = route_to_gdf(G, safety_route, weight_used="safety_cost")
    
    # Initialize Interactive Base Map
    # Use the coordinates of the start node to center the map
    start_node = shortest_route[0]
    map_center = [G.nodes[start_node]['y'], G.nodes[start_node]['x']]
    
    # Create the Folium map object
    m = folium.Map(location=map_center, zoom_start=14, tiles="cartodbdark_matter")
    
    # Add Accident Hotspot Heatmap Layer
    if accident_df is not None and not accident_df.empty:
        # Expecting columns 'latitude' and 'longitude' in accident_df
        heat_data = accident_df[['latitude', 'longitude']].values.tolist()
        HeatMap(heat_data, radius=15, blur=10, name="Accident Hotspots").add_to(m)
    
    # Determine maximum accident count for color scaling
    max_accidents = max(
        shortest_gdf['accident_count_50m'].max(), 
        safety_gdf['accident_count_50m'].max()
    )
    max_accidents = max_accidents if max_accidents > 0 else 10
    
    # Create a color map for accident risk levels
    risk_cmap = cm.LinearColormap(
        colors=['#00FF00', '#ADFF2F', '#FFFF00', '#FFA500', '#FF0000'], 
        vmin=0, 
        vmax=max_accidents,
        caption="Risk Level" 
    )
    m.add_child(risk_cmap)

    # Plot Shortest Route
    folium.GeoJson(
        shortest_gdf,
        name="Shortest Route (Dashed Line)",
        style_function=lambda feature: {
            "color": risk_cmap(feature['properties']['accident_count_50m']),
            "weight": 5,
            "opacity": 0.9,
            "dashArray": "5, 10"
        },
        tooltip=folium.GeoJsonTooltip(
            fields=["length", "safety_cost", "accident_count_50m", "accident_risk_norm"],
            aliases=["Length (m):", "Safety Cost:", "Accidents (50m):", "Risk Norm:"]
        )
    ).add_to(m)
    
    # Plot Safety-Oriented Route
    folium.GeoJson(
        safety_gdf,
        name="Safety-oriented Route",
        style_function=lambda feature: {
            "color": risk_cmap(feature['properties']['accident_count_50m']),
            "weight": 5,
            "opacity": 0.9
        },
        tooltip=folium.GeoJsonTooltip(
            fields=["length", "safety_cost", "accident_count_50m", "accident_risk_norm"],
            aliases=["Length (m):", "Safety Cost:", "Accidents (50m):", "Risk Norm:"]
        )
    ).add_to(m)
    
    # Add Start and End Markers
    end_node = shortest_route[-1]
    folium.Marker(
        location=map_center,
        popup="Origin (Start)",
        icon=folium.Icon(color="blue", icon="play")
    ).add_to(m)
    
    folium.Marker(
        location=[G.nodes[end_node]['y'], G.nodes[end_node]['x']],
        popup="Destination (End)",
        icon=folium.Icon(color="red", icon="stop")
    ).add_to(m)
    
    # Save and Output Final Deliverables
    folium.LayerControl().add_to(m)
    
    return m, shortest_gdf, safety_gdf